# FraudShield v3 — Full Retrain (Google Colab)

## Before running
1. **Runtime → Change runtime type → T4 GPU**
2. Run all cells top to bottom
3. When Drive mounts → sign in with the **same** Google account that has your data

## What's new in v3 vs v2
| | v2 | v3 |
|---|---|---|
| Epochs | 4 | **5** (early stopping) |
| Learning rate | 2e-5 | **1e-5** |
| Class weights | ✗ | **✓ WeightedTrainer** |
| Indian SMS patterns | Basic | **OTP/KYC/UPI/Courier/Job fraud + Hinglish** |
| Datasets used | 2 | **8+ sources** |

## Drive paths expected
```
My Drive/fraudshield/ml/data/raw/            ← your raw CSVs (upload from local)
My Drive/fraudshield/ml/data/processed/     ← generated here
My Drive/fraudshield/ml/registry/muril-fraud-v3/  ← model saved here
```

## Step 1 — Install dependencies

In [ ]:
%%capture
!pip install transformers==4.40.0 datasets==2.19.0 accelerate scikit-learn pandas numpy torch --quiet
print('✓ Dependencies installed')

## Step 2 — Mount Google Drive & set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE      = '/content/drive/MyDrive/fraudshield/ml'
RAW       = f'{BASE}/data/raw'
PROCESSED = f'{BASE}/data/processed'
REGISTRY  = f'{BASE}/registry/muril-fraud-v3'
CKPT_DIR  = f'{BASE}/registry/checkpoints-v3'

for d in [PROCESSED, REGISTRY, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted. Checking raw data...')
raw_dirs = [
    'huggingface', 'indian-banking',
    'kaggle/banking-phishing', 'kaggle/phishing-emails',
    'kaggle/sms-spam', 'kaggle/uci-sms',
]
for d in raw_dirs:
    path = f'{RAW}/{d}'
    if os.path.exists(path):
        files = [f for f in os.listdir(path) if f.endswith('.csv')]
        print(f'  ✓ {d:<35} {len(files)} CSV(s): {files}')
    else:
        print(f'  ✗ MISSING: {path}')
print()
print('If any are MISSING, upload your ml/data/raw/ folder to Drive first.')

## Step 3 — Generate Indian synthetic SMS (v2 expanded)

In [ ]:
import pandas as pd
import random

random.seed(42)

BANKS = ['SBI','HDFC','ICICI','Axis','Kotak','PNB','BOB','Canara','Union Bank','IndusInd']
FAKE_DOMAINS = ['sbi-kyc.net','hdfc-secure.co','icici-update.org','sbi-verify.in',
                'upi-reward.co','fedex-india.net','parcel-india.co','sbi-otp-verify.in',
                'bank-kyc-update.in','prizeindia.co','work-earn-india.net']
FAKE_SENDERS = ['SBI-ALERT','SBI-KYC','HDFC-SECURE','ICICI-KYC','BANKALERT',
                'KYC-UPDATE','AXIS-KYC','UPI-REWARD','INDIAPOST','HR-JOBS']
LEGIT_SENDERS = {'SBI':'SBIBNK','HDFC':'HDFCBK','ICICI':'ICICIB','Axis':'AXISBK',
                 'Kotak':'KOTAKB','PNB':'PNBSMS','BOB':'BOBANK','Canara':'CNRBNK',
                 'Union Bank':'UBIBNK','IndusInd':'INDBNK'}
LEGIT_DOMAINS = {'SBI':'sbi.co.in','HDFC':'hdfcbank.com','ICICI':'icicibank.com',
                 'Axis':'axisbank.com','Kotak':'kotak.com','PNB':'pnbindia.in',
                 'BOB':'bankofbaroda.in','Canara':'canarabank.com',
                 'Union Bank':'unionbankofindia.co.in','IndusInd':'indusind.com'}
HELPLINES = {'SBI':'1800-11-2211','HDFC':'1800-202-6161','ICICI':'1800-1080',
             'Axis':'1800-419-5959','Kotak':'1860-266-2666','PNB':'1800-180-2222',
             'BOB':'1800-5700','Canara':'1800-425-0018',
             'Union Bank':'1800-22-2244','IndusInd':'1860-500-5004'}

def r_amt():  return f'{random.randint(49,99000):,}'
def r_bal():  return f'{random.randint(500,300000):,.2f}'
def r_acct(): return str(random.randint(1000,9999))
def r_otp():  return str(random.randint(100000,999999))
def r_ref():  return f'UPI{random.randint(10**11,10**12-1)}'
def r_date(): return f"{random.randint(1,28):02d}-{random.choice(['Jan','Feb','Mar','Apr','May','Jun'])}-26"
def r_phone(): return f'9{random.randint(10**8,10**9-1)}'
def r_bill(): return str(random.randint(199,999))

FRAUD_OTP = [
    'Your {bank} OTP is {otp}. Share with our agent to unblock your account immediately. -{sender}',
    'ALERT: {bank} Account blocked. OTP {otp} sent. Share with helpdesk agent to restore access. -{sender}',
    '{bank} SECURITY: Your OTP is {otp}. Our executive will call. Share to verify identity. -{sender}',
    'Dear customer, suspicious login on {bank}. OTP {otp} — share with fraud team now. -{sender}',
    'Important: {bank} KYC verification OTP: {otp}. Share with agent calling within 2 min. -{sender}',
    'Your {bank} UPI OTP is {otp}. Provide to bank rep to complete verification. -{sender}',
    '{bank} Alert: OTP {otp} for account reactivation. Share immediately with bank officer. -{sender}',
]
FRAUD_KYC = [
    'Dear customer, your {bank} KYC incomplete. Update at {url} or account blocked. -{sender}',
    'URGENT: {bank} requires Aadhaar re-verification. Visit {url} now or lose access. -{sender}',
    'NOTICE: {bank} account suspended in 24 hours. Complete KYC at {url}. -{sender}',
    '{bank}: PAN mismatch detected. Update at {url} to avoid freeze. Deadline 6PM. -{sender}',
    'Final Warning: {bank} KYC overdue. Update at {url} or face permanent closure. -{sender}',
    'As per RBI mandate, {bank} requires fresh KYC. Submit at {url} or UPI stops. -{sender}',
    'Dear Customer, {bank} PAN not linked. Urgently update at {url} else account closes. -{sender}',
]
FRAUD_UPI = [
    'Congratulations! You received Rs.{amt} from {bank} UPI reward. Claim at {url}. -{sender}',
    'UPI wallet: Rs.{amt} waiting for you. Click {url} to transfer to bank. Expires 2hrs. -{sender}',
    '{bank} UPI Cashback: Rs.{amt} approved. Claim by entering UPI PIN at {url}. -{sender}',
    '{bank}: Festive bonus Rs.{amt} added to UPI. Activate at {url} with mPIN. -{sender}',
    'PM Jan Dhan: Rs.{amt} approved. Collect at {url} entering Aadhaar. -{sender}',
    'GOV SCHEME: Rs.{amt} direct benefit for {bank} customers. Claim at {url} today. -{sender}',
]
FRAUD_COURIER = [
    'Your parcel is on hold at customs. Pay Rs.49 fee at {url} to release. FEDEX India.',
    'India Post: Package (ID:{otp}) held at hub. Pay Rs.{amt} customs at {url}. -{sender}',
    'DHL Express: Shipment held. Customs fee Rs.99. Pay at {url} within 24 hrs.',
    'Amazon order held at customs. Rs.29 delivery surcharge. Pay {url} to receive today. -{sender}',
    'Blue Dart: Package held, incomplete address. Confirm & pay Rs.39 at {url}. -{sender}',
]
FRAUD_JOB = [
    'Work from home! Earn Rs.15,000/day. No experience. WhatsApp {phone}. -{sender}',
    'HIRING: Data entry job. Earn Rs.800/hr from home. Apply: {url} -{sender}',
    'Earn Rs.50,000/month part time from home. Register at {url} today. -{sender}',
    'Google hiring home-based agents. Rs.2,500/day. Register at {url}. 50 seats! -{sender}',
    'Amazon work-from-home: Earn Rs.1,200/hr reviewing products. Register at {url}. -{sender}',
]
FRAUD_EN = [
    'SECURITY ALERT from {bank}: Someone tried accessing your account. Confirm at {url}. -{sender}',
    'WARNING: {bank} will deactivate your account in 2 hours. Prevent at {url}. -{sender}',
    'Dear {bank} user, NEFT of Rs {amt} initiated. Not you? Cancel at {url}. -{sender}',
    'Congratulations! {bank} account selected for cashback Rs.{amt}. Claim at {url}. -{sender}',
    '{bank} ALERT: Multiple failed login attempts. Secure at {url}. -{sender}',
    'NOTICE: {bank} loan pre-approved Rs.5,00,000. No documents. Claim at {url}. -{sender}',
    '{bank}: Debit card XX{acct} expires soon. Renew at {url} to avoid disruption. -{sender}',
]
FRAUD_HI = [
    'ALERT: {bank} grahak, aapka account band ho jayega. Abhi verify karein: {url} -{sender}',
    'Aapka {bank} account 24 ghante mein band ho jayega. KYC update karein: {url} -{sender}',
    'Urgent: {bank} ne aapka UPI suspend kar diya. {url} par reactivate karein. -{sender}',
    'CHETAVNI: Aapke {bank} account se anajaan transaction hua. {url} par check karein. -{sender}',
    'Aapka OTP {otp} hai. {bank} agent ke saath share karein account unlock karne ke liye. -{sender}',
    'Sarkari yojana: Rs.{amt} aapke account mein aane wala hai. Link karein: {url} -{sender}',
    'Kamao Rs.15000 roz ghar baithe. Koi experience nahi chahiye. Register: {url} -{sender}',
    'Aapka parcel customs mein ruka. Rs.49 pay karein: {url} turant release. -{sender}',
    'UPI reward: Rs.{amt} aapko mila hai. Claim karne ke liye {url} par jaiyein. 1 ghante. -{sender}',
    '{bank} Alert: Naya device login detect hua. Aap nahi hain toh {url} par block karein. -{sender}',
    'Priya {bank} grahak, Rs {amt} transfer hone wala hai. Rokne ke liye {url} par jaayein. -{sender}',
]
LEGIT_BANK = [
    '{lsender}: INR {amt} debited from A/c XX{acct} on {date}. Avl Bal INR {bal}. If not you call {helpline}.',
    '{lsender}: INR {amt} credited to A/c XX{acct} on {date}. Avl Bal INR {bal}.',
    'Your {bank} OTP is {otp}. Valid for 10 mins. Do NOT share with anyone including bank officials.',
    '{lsender}: A/c XX{acct} debited INR {amt} for UPI txn on {date}. UPI Ref {ref}. Bal INR {bal}.',
    '{lsender}: Your {bank} credit card bill of INR {amt} is due on {date}. Pay at {ldomain}.',
    '{lsender}: Passbook A/c XX{acct} Bal INR {bal} as on {date}.',
    '{lsender}: Loan EMI of INR {amt} due on {date}. Auto debit from A/c XX{acct}.',
    '{lsender}: ATM txn INR {amt} on {date} from A/c XX{acct}. Bal INR {bal}.',
    'Dear {bank} customer, KYC is complete. No action required. Queries call {helpline}. -{lsender}',
    'INB:{bank} Rs.{amt} debited from A/c XX{acct} to VPA xyz@upi on {date}. Avl Bal Rs.{bal}',
    '{lsender}: NEFT of INR {amt} received in A/c XX{acct} on {date}. Avl Bal INR {bal}.',
    'OTP for your {bank} card transaction is {otp}. Valid 10 mins. Do not share with anyone.',
    '{lsender}: Your {bank} account XX{acct} linked to UPI. Txn limit Rs.1,00,000/day.',
    'Rs.{amt} Dr. from A/C XXXXXX{acct} and Cr. to nptushyent@okaxis. AvlBal:Rs{bal} -{lsender}',
    'Dear {bank} customer, your Debit Card ending {acct} renewed. Dispatched to registered address.',
]
LEGIT_TELECOM = [
    'Dear customer, your Airtel bill of Rs.{bill} is due on {date}. Pay at airtel.in.',
    'Jio: Your recharge of Rs.{bill} is successful. Validity 28 days. Data: 1.5GB/day.',
    'BSNL: Your monthly bill Rs.{bill} generated. Auto-pay on {date}. View at bsnl.co.in',
    'Vi: Your data balance is 500MB. Recharge at myvi.in to continue service.',
    'Airtel: Postpaid plan activated. Monthly Rs.{bill}. Unlimited calls + 40GB data.',
    'Dear Jio customer, your number verified. KYC complete. No further action needed.',
]
LEGIT_ECOMM = [
    'Your Amazon order #{otp} shipped. Expected delivery {date}. Track at amazon.in/orders.',
    'Flipkart: Order Rs.{amt} is out for delivery. Delivery by {date}. Track: flipkart.com.',
    'Swiggy: Your order is being prepared. Estimated delivery 30 mins.',
    'IRCTC: Ticket booked. PNR {otp}. Train departs {date}. Have a safe journey!',
    'BigBasket: Order Rs.{amt} delivered on {date} between 6AM-10AM. No action needed.',
]

rows = []
def add_fraud(templates, count, source, lang='en'):
    for _ in range(count):
        bank = random.choice(BANKS)
        try:
            text = random.choice(templates).format(
                bank=bank, url=random.choice(FAKE_DOMAINS),
                sender=random.choice(FAKE_SENDERS),
                otp=r_otp(), amt=r_amt(), acct=r_acct(), phone=r_phone())
        except KeyError:
            text = random.choice(templates)
        rows.append({'text': text, 'label': 1, 'source': source, 'language': lang})

def add_legit_bank(count):
    for _ in range(count):
        bank = random.choice(list(LEGIT_SENDERS.keys()))
        text = random.choice(LEGIT_BANK).format(
            bank=bank, lsender=LEGIT_SENDERS[bank], ldomain=LEGIT_DOMAINS[bank],
            helpline=HELPLINES[bank], amt=r_amt(), acct=r_acct(),
            bal=r_bal(), otp=r_otp(), ref=r_ref(), date=r_date())
        rows.append({'text': text, 'label': 0, 'source': 'synthetic_legit_bank', 'language': 'en'})

def add_legit_other(count):
    for _ in range(count):
        tmpl = random.choice(LEGIT_TELECOM + LEGIT_ECOMM)
        try:
            text = tmpl.format(bill=r_bill(), date=r_date(), otp=r_otp(), amt=r_amt())
        except KeyError:
            text = tmpl
        rows.append({'text': text, 'label': 0, 'source': 'synthetic_legit_other', 'language': 'en'})

add_fraud(FRAUD_OTP,     250, 'synthetic_fraud_otp')
add_fraud(FRAUD_KYC,     300, 'synthetic_fraud_kyc')
add_fraud(FRAUD_UPI,     200, 'synthetic_fraud_upi')
add_fraud(FRAUD_COURIER, 150, 'synthetic_fraud_courier')
add_fraud(FRAUD_JOB,     150, 'synthetic_fraud_job')
add_fraud(FRAUD_EN,      250, 'synthetic_fraud_en')
add_fraud(FRAUD_HI,      500, 'synthetic_fraud_hi', 'hi')
add_legit_bank(1050)
add_legit_other(350)

synthetic_df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
synthetic_out = f'{RAW}/indian-banking/synthetic_banking_sms.csv'
os.makedirs(f'{RAW}/indian-banking', exist_ok=True)
synthetic_df.to_csv(synthetic_out, index=False)
print(f'✓ {len(synthetic_df)} synthetic Indian SMS generated')
print(f'  Fraud : {int(synthetic_df.label.sum())}  |  Legit : {int((synthetic_df.label==0).sum())}')

## Step 4 — Combine all datasets

In [ ]:
import re
import json
from sklearn.model_selection import train_test_split

frames = []

def load_csv(path, text_col, label_col, label_map=None, source=None, encoding='latin-1'):
    try:
        df = pd.read_csv(path, encoding=encoding, on_bad_lines='skip')
        df = df[[text_col, label_col]].copy()
        df.columns = ['text', 'label']
        if label_map:
            df['label'] = df['label'].map(label_map)
        df['label'] = pd.to_numeric(df['label'], errors='coerce')
        df = df.dropna(subset=['label'])
        df['label'] = df['label'].astype(int)
        df = df[df['label'].isin([0, 1])]
        df['source'] = source or path
        print(f'  ✓ {(source or path):<40} {len(df):>7} rows  (fraud={df.label.sum()}, legit={(df.label==0).sum()})')
        return df[['text', 'label', 'source']]
    except Exception as e:
        print(f'  ✗ {source or path}: {e}')
        return None

def strip_pii(text):
    t = str(text)
    t = re.sub(r'\b[6-9]\d{9}\b', '[PHONE]', t)
    t = re.sub(r'XX\d{4}', 'XX[ACCT]', t)
    t = re.sub(r'\b\d{16}\b', '[CARD]', t)
    t = re.sub(r'\b[A-Z]{5}\d{4}[A-Z]\b', '[PAN]', t)
    t = re.sub(r'\b\d{12}\b', '[AADHAAR]', t)
    t = re.sub(r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}', '[EMAIL]', t)
    return t.strip()

print('── Loading datasets ──\n')

# 1. Synthetic Indian banking SMS (freshly generated)
df = load_csv(f'{RAW}/indian-banking/synthetic_banking_sms.csv', 'text', 'label', source='synthetic_indian')
if df is not None: frames.append(df)

# 2. FraudShield curated Indian dataset
df = load_csv(f'{RAW}/indian-banking/fraudshield_curated.csv', 'text', 'label', source='fraudshield_curated')
if df is not None: frames.append(df)

# 3. HuggingFace Enron
df = load_csv(f'{RAW}/huggingface/enron.csv', 'text', 'label', source='enron')
if df is not None: frames.append(df)

# 4. HuggingFace Phishing (ealvaradob) — sample to avoid domination
p = f'{RAW}/huggingface/phishing_hf.csv'
if os.path.exists(p):
    df = load_csv(p, 'text', 'label', source='hf_phishing')
    if df is not None:
        df = df.sample(n=min(40000, len(df)), random_state=42)  # cap at 40k
        frames.append(df)
        print(f'    (sampled to {len(df)} rows to avoid domination)')

# 5. HuggingFace SMS spam
df = load_csv(f'{RAW}/huggingface/sms_spam_hf.csv', 'text', 'label', source='sms_spam_hf')
if df is not None: frames.append(df)

# 6. Kaggle UCI SMS spam
for path in [f'{RAW}/kaggle/uci-sms/spam.csv', f'{RAW}/kaggle/sms-spam/SPAM text message 20170820 - Data.csv']:
    if os.path.exists(path):
        try:
            tmp = pd.read_csv(path, encoding='latin-1', on_bad_lines='skip')
            tmp.columns = [c.strip() for c in tmp.columns]
            lc = 'v1' if 'v1' in tmp.columns else ('Category' if 'Category' in tmp.columns else tmp.columns[0])
            tc = 'v2' if 'v2' in tmp.columns else ('Message' if 'Message' in tmp.columns else tmp.columns[1])
            tmp = tmp[[lc, tc]].copy()
            tmp.columns = ['label', 'text']
            tmp['label'] = tmp['label'].map({'spam': 1, 'ham': 0, 'Spam': 1, 'Ham': 0})
            tmp = tmp.dropna(subset=['label'])
            tmp['label'] = tmp['label'].astype(int)
            tmp['source'] = 'uci_sms'
            frames.append(tmp[['text', 'label', 'source']])
            print(f'  ✓ {"uci_sms":<40} {len(tmp):>7} rows')
        except Exception as e:
            print(f'  ✗ uci_sms: {e}')
        break

# 7. Kaggle phishing emails (banking-phishing folder)
for path, tc, lc, lm in [
    (f'{RAW}/kaggle/banking-phishing/phishing_email.csv', 'text', 'label', None),
    (f'{RAW}/kaggle/phishing-emails/Phishing_Email.csv', 'Email Text', 'Email Type',
     {'Phishing Email': 1, 'Safe Email': 0}),
]:
    if os.path.exists(path):
        df = load_csv(path, tc, lc, label_map=lm, source='phishing_kaggle')
        if df is not None:
            df = df.sample(n=min(30000, len(df)), random_state=42)
            frames.append(df)
        break

# 8. SpamAssassin corpus
p = f'{RAW}/kaggle/banking-phishing/SpamAssasin.csv'
if os.path.exists(p):
    df = load_csv(p, 'text', 'label', source='spamassassin')
    if df is not None:
        df = df.sample(n=min(30000, len(df)), random_state=42)
        frames.append(df)

# 9. Nigerian fraud emails
p = f'{RAW}/kaggle/banking-phishing/Nigerian_Fraud.csv'
if os.path.exists(p):
    df = load_csv(p, 'text', 'label', source='nigerian_fraud')
    if df is not None:
        df = df.sample(n=min(20000, len(df)), random_state=42)
        frames.append(df)

# Combine
print(f'\n── Combining {len(frames)} sources ──')
combined = pd.concat(frames, ignore_index=True)
print(f'Total before cleaning : {len(combined):,}')

combined['text'] = combined['text'].apply(strip_pii)
combined['text'] = combined['text'].astype(str).str.strip()
combined = combined[combined['text'].str.len() > 15]
combined = combined.drop_duplicates(subset=['text'])
combined['label'] = combined['label'].astype(int)
print(f'Total after cleaning  : {len(combined):,}  (fraud={combined.label.sum():,}, legit={(combined.label==0).sum():,})')

# Balance: keep all fraud, cap legit at 2x fraud to avoid imbalance
fraud_df = combined[combined['label'] == 1]
legit_df = combined[combined['label'] == 0]
target = min(len(legit_df), len(fraud_df) * 2)
balanced = pd.concat([fraud_df, legit_df.sample(n=target, random_state=42)])
balanced = balanced.sample(frac=1, random_state=42).reset_index(drop=True)
print(f'After balancing       : {len(balanced):,}  (fraud={balanced.label.sum():,}, legit={(balanced.label==0).sum():,})')

# 70/15/15 split
train, temp = train_test_split(balanced, test_size=0.30, random_state=42, stratify=balanced['label'])
val,   test = train_test_split(temp,     test_size=0.50, random_state=42, stratify=temp['label'])

train.to_csv(f'{PROCESSED}/train_v3.csv', index=False)
val.to_csv(  f'{PROCESSED}/val_v3.csv',   index=False)
test.to_csv( f'{PROCESSED}/test_v3_LOCKED.csv', index=False)

print(f'\n── Split ──')
print(f'  Train : {len(train):,}')
print(f'  Val   : {len(val):,}')
print(f'  Test  : {len(test):,}  ← LOCKED')
print(f'\nSource breakdown:')
for src, cnt in combined['source'].value_counts().items():
    print(f'  {cnt:>8}  {src}')

## Step 5 — Tokenize

In [ ]:
import numpy as np
import torch
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = 'google/muril-base-cased'
MAX_LEN    = 256

print(f'GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY — switch to T4 GPU in Runtime"}')

train_df = pd.read_csv(f'{PROCESSED}/train_v3.csv')
val_df   = pd.read_csv(f'{PROCESSED}/val_v3.csv')
print(f'Train : {len(train_df):,}  (fraud={int(train_df.label.sum()):,}  legit={int((train_df.label==0).sum()):,})')
print(f'Val   : {len(val_df):,}')

print('\nLoading MuRIL tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_pandas(train_df[['text', 'label']].astype({'label': int}))
val_ds   = Dataset.from_pandas(val_df[['text',   'label']].astype({'label': int}))

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN, padding='max_length')

print('Tokenizing...')
train_ds = train_ds.map(tokenize, batched=True, batch_size=512)
val_ds   = val_ds.map(  tokenize, batched=True, batch_size=512)
train_ds = train_ds.rename_column('label', 'labels')
val_ds   = val_ds.rename_column(  'label', 'labels')
train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
val_ds.set_format(  'torch', columns=['input_ids', 'attention_mask', 'labels'])
print('✓ Tokenization complete')

## Step 6 — Train MuRIL v3 (WeightedTrainer, EPOCHS=5, LR=1e-5)

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)
from torch.nn import CrossEntropyLoss
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

BATCH_SIZE = 16
EPOCHS     = 5      # was 4 — early stopping prevents overfit
LR         = 1e-5   # was 2e-5 — MuRIL is sensitive to LR

# Class weights — penalize missing fraud more
fraud_count = int(train_df['label'].sum())
legit_count = len(train_df) - fraud_count
ratio = legit_count / fraud_count
print(f'Class ratio legit/fraud = {ratio:.2f} → weight = [1.0, {ratio:.2f}]')

weight = torch.tensor([1.0, ratio], dtype=torch.float)
if torch.cuda.is_available():
    weight = weight.cuda()

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        loss    = CrossEntropyLoss(weight=weight)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy' : round(accuracy_score( labels, preds), 4),
        'precision': round(precision_score(labels, preds, zero_division=0), 4),
        'recall'   : round(recall_score(   labels, preds, zero_division=0), 4),
        'f1'       : round(f1_score(       labels, preds, zero_division=0), 4),
    }

print('Loading MuRIL model...')
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

args = TrainingArguments(
    output_dir                  = CKPT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    learning_rate               = LR,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    greater_is_better           = True,
    logging_steps               = 100,
    fp16                        = torch.cuda.is_available(),
    report_to                   = 'none',
    save_total_limit            = 2,
)

trainer = WeightedTrainer(
    model           = model,
    args            = args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print()
print('Training v3 started...')
print(f'  Epochs : {EPOCHS} max  (early stopping patience=2)')
print(f'  LR     : {LR}')
print(f'  Class weights : [1.0, {ratio:.2f}]')
print('  Checkpoints saved to Drive each epoch')
print('  Estimated: ~60-90 min on T4 GPU')
print()
trainer.train()

## Step 7 — Evaluate on validation set

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

pred_out = trainer.predict(val_ds)
preds    = np.argmax(pred_out.predictions, axis=1)
labels   = val_ds['labels']

tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

print('── Validation Results (v3) ──')
print(f'  Accuracy  : {accuracy_score(labels, preds):.4f}')
print(f'  Precision : {precision_score(labels, preds):.4f}')
print(f'  Recall    : {recall_score(labels, preds):.4f}')
print(f'  F1        : {f1_score(labels, preds):.4f}')
print(f'  FP rate   : {fpr:.4f}  (target < 0.02)')
print(f'  TP={tp}  FP={fp}  TN={tn}  FN={fn}')
print()
print(classification_report(labels, preds, target_names=['Legit', 'Fraud']))

## Step 8 — Real SMS test (Indian banking messages)

In [ ]:
test_messages = [
    # === LEGIT — must NOT be flagged ===
    ('Dear BOB UPI User: Your account is credited with INR 145.00 on 2026-03-23 by UPI Ref No 195310192638; AvlBal: Rs641.94 - BOB',
     0, 'BOB UPI credit — LEGIT'),
    ('Rs.500.00 Dr. from A/C XXXXXX5608 and Cr. to nptushyent@okaxis. Ref:644933179447. AvlBal:Rs121.94. Not you? Call 18005700-BOB',
     0, 'BOB debit @okaxis — LEGIT'),
    ('Rs.130.00 Dr. from A/C XXXXXX5608 and Cr. to nptushyent@okicici. Ref:530359835274. AvlBal:Rs2054.30. Not you? Call 18005700-BOB',
     0, 'BOB debit @okicici — LEGIT'),
    ('HDFCBK: OTP for HDFC Net Banking login is 847293. Valid for 10 mins. Do NOT share with anyone.',
     0, 'HDFC real OTP — LEGIT'),
    ('AXISBK: INR 2500.00 credited to A/c XX4139 on 15-01-26. Avl Bal INR 45,200.00',
     0, 'Axis credit — LEGIT'),
    ('Dear customer, your Airtel bill of Rs.399 is due on 31-Mar. Pay at airtel.in',
     0, 'Airtel bill — LEGIT'),
    ('Your Amazon order #503-12345 has been shipped. Expected delivery: 28-Mar. Track at amazon.in/orders.',
     0, 'Amazon delivery — LEGIT'),
    ('SBIBNK: INR 1500 debited from A/c XX4521 to VPA xyz@upi on 27-Mar-26. Avl Bal Rs.8420.15',
     0, 'SBI real UPI debit — LEGIT'),
    # === FRAUD — must be caught ===
    ('SBI-KYC: Your account will be blocked. Verify at sbi-kyc.net immediately or permanently closed.',
     1, 'SBI KYC scam — FRAUD'),
    ('Congratulations! Rs.500000 RBI Lucky Draw. Claim at rbi-lucky.net within 24hrs.',
     1, 'Lottery scam — FRAUD'),
    ('Work from home! Earn Rs.45000/month. Pay Rs.499 registration at gpay-alert.co/jobs.',
     1, 'Job scam — FRAUD'),
    ('Your SBI OTP is 847291. Share with our agent to unblock your account immediately. -SBI-ALERT',
     1, 'OTP share scam — FRAUD'),
    ('HDFC-SECURE: Suspicious activity on your account. Verify at hdfc-secure.co immediately.',
     1, 'HDFC fake security — FRAUD'),
    ('Your parcel is held at customs. Pay Rs.49 at fedex-india.net to release. FEDEX India.',
     1, 'Courier scam — FRAUD'),
    ('Aapka SBI account band ho jayega. Abhi verify karein: sbi-verify.in -SBI-KYC',
     1, 'Hindi KYC scam — FRAUD'),
    ('You received Rs.5000 UPI reward from HDFC. Click upi-reward.co to claim. Expires 1hr.',
     1, 'UPI reward scam — FRAUD'),
]

model.eval()
correct = 0
print('── Real SMS Test (v3) ──\n')
for text, true_label, description in test_messages:
    inputs = tokenizer(text, return_tensors='pt', truncation=True,
                       max_length=MAX_LEN, padding='max_length')
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    pred      = torch.argmax(logits, dim=1).item()
    conf      = torch.softmax(logits, dim=1).max().item()
    status    = '✓ PASS' if pred == true_label else '✗ FAIL'
    label_str = 'FRAUD' if pred == 1 else 'LEGIT'
    if pred == true_label: correct += 1
    print(f'[{status}] [{label_str}] {conf:.1%}  {description}')
    print(f'  "{text[:95]}"')
    print()

print(f'Score: {correct}/{len(test_messages)} correct')
if correct == len(test_messages):
    print('✓ All passing — safe to save')
else:
    failed = len(test_messages) - correct
    print(f'⚠ {failed} failure(s) — review dataset or add more examples')

## Step 9 — Save model to Drive

In [ ]:
import json

model.save_pretrained(REGISTRY)
tokenizer.save_pretrained(REGISTRY)

results = {
    'model_version'       : 'muril-fraud-v3',
    'base_model'          : MODEL_NAME,
    'train_size'          : len(train_df),
    'val_size'            : len(val_df),
    'accuracy'            : round(accuracy_score( labels, preds), 4),
    'precision'           : round(precision_score(labels, preds), 4),
    'recall'              : round(recall_score(   labels, preds), 4),
    'f1'                  : round(f1_score(       labels, preds), 4),
    'false_positive_rate' : round(fpr, 4),
    'epochs_config'       : EPOCHS,
    'learning_rate'       : LR,
    'class_weights'       : [1.0, round(ratio, 3)],
    'sms_test_score'      : f'{correct}/{len(test_messages)}',
}

with open(f'{REGISTRY}/val_results_v3.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'✓ Model saved → {REGISTRY}')
print()
print('Results:')
for k, v in results.items():
    print(f'  {k:<25}: {v}')
print()
print('Files saved:')
for f in sorted(os.listdir(REGISTRY)):
    size_mb = os.path.getsize(f'{REGISTRY}/{f}') / 1024 / 1024
    print(f'  {f:<45} {size_mb:.1f} MB')
print()
print('Done. Deploy: copy muril-fraud-v3/ into services/ai/model_registry/')